# Imitation Training — EdgeTransformerNet

Trains the `EdgeTransformerNet` on winner-action replay data.
Output: `best_model.pt` — used as the actor for PPO fine-tuning.

## Required Kaggle inputs

| Dataset slug | Contents | How to create |
|---|---|---|
| `orbitswar-src` | Source code zip | `python zip_kaggle_dataset.py --src` |
| `orbitswar-dataset` | `dataset.npz` (float16 uncompressed) | Run `convert_dataset_f16.py`, rename output to `dataset.npz`, then `python zip_kaggle_dataset.py --dataset` |

`dataset.npz` is float16 uncompressed — `mmap_mode='r'` keeps X on disk during training (zero RAM spike).

## Output
- `/kaggle/working/checkpoints/best_model.pt` — best val-loss checkpoint
- `/kaggle/working/checkpoints/latest_model.pt` — last epoch checkpoint
- `/kaggle/working/checkpoints/training_history.json` — loss / F1 / entropy per epoch

Download all three from the **Output** tab.  
Then zip `best_model.pt` as `orbitswar-imitation` and attach to `train_transformer_ppo.ipynb`.

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 1 — GPU check
# ─────────────────────────────────────────────────────────
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU — training will be very slow (~20 min/epoch).')
    print('         Enable GPU: Settings -> Accelerator -> GPU T4 x2')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 2b — Logging setup
# Appends to previous log if one exists in orbitswar-imitation
# dataset — so the full training history accumulates across
# Kaggle sessions. Download imitation_training.log each time.
# ─────────────────────────────────────────────────────────
import sys, time, shutil
from pathlib import Path

LOG_PATH = Path('/kaggle/working/imitation_training.log')

# Check for previous log from the imitation dataset (resume accumulation)
_prev_log = Path('/kaggle/input/orbitswar-imitation/imitation_training.log')
if _prev_log.exists() and not LOG_PATH.exists():
    shutil.copy2(_prev_log, LOG_PATH)
    _log_mode = 'a'
    print(f'Previous log found ({_prev_log.stat().st_size/1024:.1f} KB) — appending.')
else:
    _log_mode = 'w'


class _TeeLogger:
    """Duplicate every print() to a persistent log file."""
    def __init__(self, path):
        self._orig = sys.stdout
        self._file = open(path, _log_mode, buffering=1, encoding='utf-8')
        ts = time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())
        if _log_mode == 'a':
            self._file.write(f'\n\n# ══ Resumed session : {ts} ══\n\n')
        else:
            self._file.write('# OrbitsWar — Imitation Training Log\n')
            self._file.write(f'# Started : {ts}\n\n')
        self._file.flush()

    def write(self, msg):
        self._orig.write(msg)
        try:   self._file.write(msg)
        except: pass

    def flush(self):
        self._orig.flush()
        try:   self._file.flush()
        except: pass

    def isatty(self):
        return False

    def close(self):
        sys.stdout = self._orig
        try:   self._file.flush(); self._file.close()
        except: pass


_tee = _TeeLogger(LOG_PATH)
sys.stdout = _tee
print(f'Logging to : {LOG_PATH}  (mode={_log_mode})')
print('(All print() output is mirrored here — download from Output tab.)')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 2 — Dataset slugs
# ─────────────────────────────────────────────────────────
from pathlib import Path

SRC_DATASET  = 'orbitswar-src'
DATA_DATASET = 'orbitswar-dataset'

SRC_DIR  = Path(f'/kaggle/input/{SRC_DATASET}')
DATA_DIR = Path(f'/kaggle/input/{DATA_DATASET}')

# Prefer float16 uncompressed (mmap-compatible, zero RAM for X)
# Falls back to dataset.npz if f16 not available
DATASET_NPZ = DATA_DIR / 'dataset_f16.npz'
if not DATASET_NPZ.exists():
    DATASET_NPZ = DATA_DIR / 'dataset.npz'

OUT_CKPT_DIR = Path('/kaggle/working/checkpoints')

print(f'Source  : {SRC_DIR}  exists={SRC_DIR.exists()}')
print(f'Dataset : {DATASET_NPZ}  exists={DATASET_NPZ.exists()}')

if not SRC_DIR.exists():
    raise FileNotFoundError(f'Source dataset not found: {SRC_DIR}')
if not DATASET_NPZ.exists():
    raise FileNotFoundError(
        f'Dataset not found in {DATA_DIR}\n'
        'Expected dataset_f16.npz or dataset.npz\n'
        'Attach the orbitswar-dataset Kaggle dataset to this notebook.'
    )

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 3 — Set up Python path (no file copying needed)
# ─────────────────────────────────────────────────────────
import sys

for p in [
    str(SRC_DIR / 'imitation'),   # model.py, features.py
    str(SRC_DIR),                  # main.py (needed by features.py)
    str(SRC_DIR / 'classifier'),   # train_from_replays.py
]:
    if p not in sys.path:
        sys.path.insert(0, p)

# Verify model import
from model import EdgeTransformerNet
n = sum(p.numel() for p in EdgeTransformerNet().parameters())
print(f'EdgeTransformerNet params: {n:,}')
print('Imports OK.')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 4 — Inspect dataset
# mmap_mode='r' keeps X on disk (zero RAM).
# NaN check uses a contiguous slice — random mmap seeks over
# 500k rows would be slow; sequential reads use disk prefetch.
# ─────────────────────────────────────────────────────────
import numpy as np

d          = np.load(DATASET_NPZ, mmap_mode='r')
X          = d['X']            # float16 memmap — stays on disk
y_action   = d['y_action'][:]  # small arrays — load fully
y_fraction = d['y_fraction'][:]
n_players  = d['n_players'][:] if 'n_players' in d else None

print(f'Total samples  : {len(X):,}')
print(f'Feature dim    : {X.shape[1]}')
print(f'Dtype          : {X.dtype}')
print(f'Positives      : {y_action.sum():,}  ({y_action.mean()*100:.1f}%)')
print(f'Negatives      : {(y_action==0).sum():,}')
if n_players is not None:
    n1v1 = (n_players == 2).sum()
    n4p  = (n_players == 4).sum()
    print(f'1v1 samples    : {n1v1:,}  ({n1v1/len(X)*100:.1f}%)')
    print(f'4p FFA samples : {n4p:,}  ({n4p/len(X)*100:.1f}%)')

# Check first 500k rows sequentially (fast sequential mmap read)
_sample = X[:500_000]
print(f'NaN/Inf in X (first 500k rows): {(~np.isfinite(_sample)).sum()}')
del _sample

# Ship fraction distribution (positives only)
fracs = y_fraction[~np.isnan(y_fraction)]
print(f'\nShip fractions (positives):')
print(f'  min={fracs.min():.3f}  p25={np.percentile(fracs,25):.3f}  '
      f'median={np.median(fracs):.3f}  p75={np.percentile(fracs,75):.3f}  max={fracs.max():.3f}')

# Free everything — model.py opens its own mmap handle
del d, X, y_action, y_fraction, n_players, fracs

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 5 — Training hyperparameters
# ─────────────────────────────────────────────────────────

# Model architecture
D_MODEL    = 128
NHEAD      = 4
NUM_LAYERS = 3
DROPOUT    = 0.10

# Training
BATCH_SIZE = 128
EPOCHS     = 40
LR         = 3e-4

# Loss weights
POS_WEIGHT      = 2.0
FRACTION_WEIGHT = 0.5

# Set to a checkpoint path to resume from a previous run; None = fresh start
PRETRAINED = None

# ── Accurate time estimate ────────────────────────────────
import numpy as np

# mmap_mode='r' — only replay/step arrays are accessed, X never touched
d_tmp      = np.load(str(DATASET_NPZ), mmap_mode='r')
replay_ids = d_tmp['replay'][:]
step_ids   = d_tmp['step'][:]
del d_tmp

turn_ids   = replay_ids.astype('int64') * 10_000 + step_ids.astype('int64')
n_turns    = len(set(turn_ids))
n_train    = int(n_turns * 0.9)
batches_ep = n_train // BATCH_SIZE
del replay_ids, step_ids, turn_ids

T4_BATCHES_PER_MIN = 1000
min_per_epoch = batches_ep / T4_BATCHES_PER_MIN
total_hours   = min_per_epoch * EPOCHS / 60

print(f'Dataset       : {DATASET_NPZ}')
print(f'Turns         : {n_turns:,}  →  {n_train:,} train turns')
print(f'Batches/epoch : {batches_ep:,}')
print(f'Est. per epoch: ~{min_per_epoch:.0f} min on T4')
print(f'Est. total    : ~{total_hours:.1f}h for {EPOCHS} epochs')
if total_hours > 11:
    print(f'WARNING: reduce EPOCHS to {int(10.5 * 60 / min_per_epoch)} to stay under 11h.')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 6 — Train
# ─────────────────────────────────────────────────────────
from model import train as imitation_train

OUT_CKPT_DIR.mkdir(parents=True, exist_ok=True)

model, history = imitation_train(
    dataset_path         = str(DATASET_NPZ),
    out_dir              = str(OUT_CKPT_DIR),
    d_model              = D_MODEL,
    nhead                = NHEAD,
    num_layers           = NUM_LAYERS,
    dropout              = DROPOUT,
    batch_size           = BATCH_SIZE,
    lr                   = LR,
    epochs               = EPOCHS,
    pos_weight_factor    = POS_WEIGHT,
    fraction_loss_weight = FRACTION_WEIGHT,
    device               = DEVICE,
    pretrained_path      = PRETRAINED,
)

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 7 — Plot training curves
# ─────────────────────────────────────────────────────────
import json
import matplotlib.pyplot as plt

hist_path = OUT_CKPT_DIR / 'training_history.json'
if hist_path.exists():
    with open(hist_path) as f:
        hist = json.load(f)
else:
    hist = history

epochs_x  = [h['epoch']    for h in hist]
tr_loss   = [h['tr_loss']  for h in hist]
val_loss  = [h['val_loss'] for h in hist]
entropy   = [h.get('entropy', float('nan')) for h in hist]
f1        = [h['f1']       for h in hist]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs_x, tr_loss,  label='train loss')
axes[0].plot(epochs_x, val_loss, label='val loss')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].set_xlabel('Epoch')
axes[0].axhline(0.45, color='g', linestyle='--', alpha=0.5, label='target ~0.45')
axes[0].legend()

axes[1].plot(epochs_x, entropy, color='orange', label='val entropy')
axes[1].set_title('Action Entropy (lower = more confident)')
axes[1].set_xlabel('Epoch')
axes[1].axhline(0.693, color='r', linestyle='--', alpha=0.4, label='max (random)')
axes[1].axhline(0.20,  color='g', linestyle='--', alpha=0.4, label='target ~0.20')
axes[1].legend()

axes[2].plot(epochs_x, f1, color='purple', label='val F1')
axes[2].set_title('F1 Score (higher = better)')
axes[2].set_xlabel('Epoch')
axes[2].axhline(0.65, color='g', linestyle='--', alpha=0.4, label='target ~0.65')
axes[2].legend()

plt.suptitle(f'Imitation Training — EdgeTransformerNet  '
             f'(best val={min(val_loss):.4f} @ epoch {epochs_x[val_loss.index(min(val_loss))]})',
             fontsize=11, y=1.02)
plt.tight_layout()

# Save to both locations so it appears in the Output tab
# alongside model checkpoints AND at the working root
CURVE_WORKING  = Path('/kaggle/working/training_curves.png')
CURVE_CKPT_DIR = OUT_CKPT_DIR / 'training_curves.png'

plt.savefig(CURVE_WORKING,  dpi=120, bbox_inches='tight')
plt.savefig(CURVE_CKPT_DIR, dpi=120, bbox_inches='tight')
plt.show()

print(f'Saved training_curves.png')
print(f'  -> {CURVE_WORKING}')
print(f'  -> {CURVE_CKPT_DIR}')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 8 — Final summary + output files
# ─────────────────────────────────────────────────────────
best_epoch = min(history, key=lambda h: h['val_loss'])
print(f'Best epoch : {best_epoch["epoch"]}')
print(f'  val_loss : {best_epoch["val_loss"]:.4f}')
print(f'  entropy  : {best_epoch.get("entropy", float("nan")):.4f}')
print(f'  F1       : {best_epoch["f1"]:.4f}')
print(f'  Precision: {best_epoch["precision"]:.4f}')
print(f'  Recall   : {best_epoch["recall"]:.4f}')

print('\nOutput files (download from Output tab):')
for p in sorted(OUT_CKPT_DIR.iterdir()):
    print(f'  checkpoints/{p.name}  ({p.stat().st_size/1e6:.2f} MB)')
log_sz = LOG_PATH.stat().st_size / 1024 if LOG_PATH.exists() else 0
print(f'  imitation_training.log  ({log_sz:.1f} KB)  ← accumulates across sessions')

print()
print('Next steps:')
print('  1. Download these files from the Output tab:')
print('       best_model.pt')
print('       imitation_training.log')
print('  2. Place them locally:')
print('       best_model.pt        → imitation/checkpoints/best_model.pt')
print('       imitation_training.log → imitation/checkpoints/imitation_training.log')
print('  3. Run: python zip_kaggle_dataset.py --models')
print('     → orbitswar-imitation.zip includes both model + log')
print('  4. Upload orbitswar-imitation.zip to Kaggle')
print('  5. Run train_transformer_ppo.ipynb for PPO fine-tuning')

# ── Flush & close log ─────────────────────────────────────
try:
    import time as _t
    print(f'\n# Log ended : {_t.strftime("%Y-%m-%d %H:%M:%S UTC", _t.gmtime())}')
    _tee.flush()
    _log_sz = LOG_PATH.stat().st_size
    _tee.close()
    print(f'\nLog saved : {LOG_PATH}  ({_log_sz / 1024:.1f} KB total across all sessions)')
except Exception as _log_err:
    print(f'Log close error: {_log_err}')